# INTEGRACION DATA SISMO - CALI - AGOSTO 2026

## 1. Aprovisionamiento

In [ ]:
%pip install -q gspread google-auth
%pip install -q rapidfuzz scikit-learn model2vec

## 2. Importaciones

In [ ]:
import pandas as pd
import gspread
from google.oauth2.service_account import Credentials
import random
import string

## 3. Funciones

### 3.1. Normalización de direcciones

In [ ]:
import re

def normalize_address(address) -> str:
    """
    Normalizes Colombian cadastral addresses to IGAC standard (Circular 300/01).

    Output format: [TIPO_VÍA] [NÚMERO] # [NÚMERO_GENERADORA]-[PLACA], Complement
    Example: "Calle 80 No. 45-23, barrio el peñón"  →  "CL 80 # 45-23, Barrio El Peñón"
             "Kra 10 Num 15-20, apto 301"            →  "KR 10 # 15-20, Apto 301"

    Title case is applied only to the complement (text after the first comma).
    The nomenclature part (address codes + numbers) stays in uppercase.

    Source: IGAC Instructivo para Direcciones (Circular 300/01 / Resolución MEN 166/04)
    Official abbreviations: CL, KR, AV, AC, AK, DG, TV, AU, BL, CT, CQ, CV, CC, PJ, PS, PT, TC, VT, VI
    Note: official Carrera code is KR (not CR).
    """
    if address is None:
        return ""

    value = str(address).strip()
    if not value or value in {"-", " "}:
        return value

    s = value.upper()

    # Pre-process: strip internal dots from abbreviations  →  K.R.A. → KRA, C.L. → CL
    s = re.sub(
        r'\b([A-Z])\.([A-Z])(?:\.([A-Z]))?\.?',
        lambda m: m.group(1) + m.group(2) + (m.group(3) or ""),
        s,
    )

    # ── Road type → IGAC standard code ───────────────────────────────────────
    # Order matters: compound types before their components;
    # full words before short abbreviations.
    _ROAD_TYPES = [
        # Compound avenidas (must precede plain AV, CL, KR)
        (r'\bAVENIDA CALLE\b|\bAV CALLE\b|\bAV CL\b',                          'AC'),
        (r'\bAVENIDA K?ARRERA\b|\bAV K?ARRERA\b|\bAV K?R\b|\bAV CRA\b',        'AK'),
        # Autopista (AU)
        (r'\bAUTOPISTA\b|\bAUTOP\b|\bAUT\b',                                   'AU'),
        # Avenida (AV)
        (r'\bAVENIDA\b|\bAVDA\b|\bAVD\b|\bAVE\b|\bAV\b',                       'AV'),
        # Carretera (CT) — before Carrera to avoid CARR collision
        (r'\bCARRETERA\b|\bCARRET\b',                                           'CT'),
        # Carrera (KR) — official IGAC code is KR, not CR
        (r'\bCARRERA\b|\bKARRERA\b|\bCARR\b|\bCRA\b|\bKRA\b|\bKRR\b|\bKR\b|\bCR\b', 'KR'),
        # Calle (CL)
        (r'\bCALLE\b|\bCALL\b|\bCLLE\b|\bCLL\b|\bCL\b',                        'CL'),
        # Circunvalar (CV) — before Circular to avoid CQ collision
        (r'\bCIRCUNVALAR\b|\bCIRCUNV\b|\bCIRCV\b',                            'CV'),
        # Circular (CQ)
        (r'\bCIRCULAR\b|\bCIRC\b',                                             'CQ'),
        # Diagonal (DG)
        (r'\bDIAGONAL\b|\bDIAG\b|\bDG\b',                                      'DG'),
        # Transversal (TV)
        (r'\bTRANSVERSAL\b|\bTRANSV\b|\bTRANS\b|\bTRAV\b|\bTV\b|\bTR\b',      'TV'),
        # Troncal (TC)
        (r'\bTRONCAL\b|\bTRONC\b',                                              'TC'),
        # Bulevar (BL)
        (r'\bBULEVAR\b|\bBOULEVAR\b|\bBLVD\b|\bBL\b',                        'BL'),
        # Pasaje (PJ) — before Paseo
        (r'\bPASAJE\b|\bPSJE\b|\bPJE\b|\bPJ\b',                               'PJ'),
        # Paseo (PS)
        (r'\bPASEO\b|\bPSO\b',                                                  'PS'),
        # Peatonal (PT)
        (r'\bPEATONAL\b|\bPEAT\b',                                             'PT'),
        # Variante (VT)
        (r'\bVARIANTE\b',                                                        'VT'),
        # Vía (VI)
        (r'\bV[IÍ]A\b',                                                          'VI'),
        # Cuentas Corridas (CC)
        (r'\bCUENTAS? CORRIDAS?\b',                                             'CC'),
    ]
    for pattern, code in _ROAD_TYPES:
        s = re.sub(pattern, code, s)

    # Strip stray trailing dots left after code substitution  →  "CL." → "CL"
    s = re.sub(
        r'\b(AU|AV|AC|AK|KR|CL|CV|CQ|CT|DG|TV|TC|BL|PJ|PS|PT|VT|VI|CC)\.',
        r'\1',
        s,
    )

    # ── Número sign (# separator) ─────────────────────────────────────────────
    # Word-bounded variants: NUMERO, NUM, NRO, NO, NN
    # (?![A-ZÁÉÍÓÚ]) guards against eating the start of a longer word:
    # without it, "NORTE-2" → "# RTE-2" because \bNO matches inside NORTE
    s = re.sub(
        r'\b(?:N[ÚU]MERO|NUMERO|N[ÚU]M|NUM|NRO|NO|NN)(?![A-ZÁÉÍÓÚ])\.?\s*',
        '# ',
        s,
        flags=re.UNICODE,
    )
    # Non-word variants: N°, Nº  (no \b needed around the degree/ordinal sign)
    s = re.sub(r'N[°º]\.?\s*', '# ', s)

    # ── Whitespace normalization ──────────────────────────────────────────────
    s = re.sub(r'\s*#\s*', ' # ', s)   # exactly one space on each side of #
    s = re.sub(r'\s+', ' ', s).strip()

    # ── Casing: nomenclature stays uppercase; complement (after comma) → title ─
    if ',' in s:
        nomenclatura, complement = s.split(',', 1)
        return nomenclatura.strip() + ', ' + complement.strip().title()
    return s

### 3.2. Handshake y Vectorización

In [ ]:
import numpy as np

# ── Handshake (canonical fingerprint string) ──────────────────────────────────
_HANDSHAKE_RE = re.compile(
    r'^(AU|AV|AC|AK|KR|CL|CV|CQ|CT|DG|TV|TC|BL|PJ|PS|PT|VT|VI|CC)'
    r'\s+(\d+[A-Z]?(?:\s?BIS)?)'
    r'(?:\s+#\s+(\d+[A-Z]?)(?:[-\s](\d+))?)?',
    re.IGNORECASE,
)

def make_handshake(address) -> str:
    """
    Extracts a compact canonical fingerprint from a normalized Colombian address.
    Strips complement (text after comma) — only the nomenclature part is used.

    Examples:
      "CL 5 # 43A-15, El Lido, Cali"          →  "CL5#43A-15"
      "KR 57 # 3-88, Cuarto De Legua, Cali"   →  "KR57#3-88"
      "CL 10 BIS # 70 34, El Limonar, Cali"   →  "CL10BIS#70-34"
      "CL 3 # 61B, Pampalinda, Cali"           →  "CL3#61B"
      "KR 72 CON CL 11, La Hacienda, Cali"     →  "KR72"
    """
    if not address or str(address).strip() in {"", "-"}:
        return ""
    nomenclatura = str(address).split(",")[0].strip().upper()
    m = _HANDSHAKE_RE.match(nomenclatura)
    if m:
        road_type = m.group(1)
        road_num  = m.group(2).replace(" ", "")
        cross_num = m.group(3) or ""
        distance  = m.group(4) or ""
        if cross_num and distance:
            return f"{road_type}{road_num}#{cross_num}-{distance}"
        if cross_num:
            return f"{road_type}{road_num}#{cross_num}"
        return f"{road_type}{road_num}"
    return re.sub(r'\s+', '', nomenclatura)

# ── Vector representation (3D) — three-pass parser ───────────────────────────

# Pass 1: road type + number + optional suffix (letter OR BIS, with optional space)
_ROAD_HEAD_RE = re.compile(
    r'^(AU|AV|AC|AK|KR|CL|CV|CQ|CT|DG|TV|TC|BL|PJ|PS|PT|VT|VI|CC)\s+'
    r'(\d+)\s*(BIS\b|[A-Z](?![A-Z]))?',
    re.IGNORECASE,
)

# Pass 2a: first valid "# NUMBER [LETTER] [-+ DISTANCE]" anywhere in the remainder
# skips qualifier tokens (NTE., RTE, SUR, CON, etc.) that appear right after #
# [-\s]+ handles multi-char separators like " - " (space-dash-space)
_CROSS_RE = re.compile(
    r'#\s*(?:[A-Z][A-Z0-9]*\.?\s+)*(\d+)\s*([A-Z](?![A-Z]))?(?:[-\s]+(\d+))?',
    re.IGNORECASE,
)

# Pass 2b: if cross found but distance still missing, extract from "# QUALIFIER-NUMBER"
# e.g. "# RTE-2" → distance=2
_DIST_QUALIFIER_RE = re.compile(
    r'#\s*[A-Z][A-Z0-9]*\.?-(\d+)',
    re.IGNORECASE,
)

# Pass 2c: no # at all — parse implicit "CROSS [LETTER] [DIGIT] DISTANCE" sequence
# absorbs embedded digit in cross (e.g. 1A9 → cross=1A, noise=9, distance=next token)
_CROSS_NO_HASH_RE = re.compile(
    r'(?<!\w)(\d{1,3})\s*([A-Z](?![A-Z]))?\d*[-\s]+(\d{1,3})(?!\d)',
    re.IGNORECASE,
)

_ROAD_CODES = {
    'CL':1,'KR':2,'AV':3,'AC':4,'AK':5,'DG':6,'TV':7,'AU':8,'BL':9,
    'CT':10,'CQ':11,'CV':12,'CC':13,'PJ':14,'PS':15,'PT':16,'TC':17,'VT':18,'VI':19,
}

def _to_decimal(num_str, letter=None, bis=False) -> float:
    """
    5    → 5.0
    5A   → 5.01   (A=+0.01, B=+0.02, ..., Z=+0.26)
    5 BIS→ 5.50   (BIS = mid-point between N and N+1)
    """
    if not num_str:
        return 0.0
    base = float(num_str)
    if bis:
        return base + 0.50
    if letter:
        return base + (ord(letter.upper()) - ord('A') + 1) * 0.01
    return base

def address_to_vector(address) -> np.ndarray:
    """
    Converts a normalized Colombian address to a 3D vector.

    Dimensions:
      [0] road_full  = road_type_code * 1000 + road_num_decimal
                       CL 5A → 1005.01  |  KR 60A → 2060.01
      [1] cross_full = cross_num_decimal
                       43A → 43.01  |  61B → 61.02
      [2] distance   = placa distance from corner

    Euclidean distance semantics:
      Different road type   → Δ ≥ 1000   (completely different road)
      Adjacent road number  → Δ = 1      (one block)
      Letter suffix         → Δ = 0.01   (sub-block subdivision)
      BIS suffix            → Δ = 0.50   (mid-block duplicate)

    Three-pass parsing strategy:
      1. Road head from start (type + num + letter/BIS, space-tolerant)
      2a. Search for "# NUMBER" — skips qualifier tokens (NTE, RTE, CON, etc.)
          [-\s]+ handles separators like " - " (space-dash-space)
      2b. If distance still missing, extract from "# QUALIFIER-NUMBER" (e.g. # RTE-2)
      2c. No # found — fallback to implicit CROSS DISTANCE sequence (e.g. 1A9 80)
    """
    null_vec = np.zeros(3, dtype=np.float32)
    if not address or str(address).strip() in {"", "-"}:
        return null_vec
    nomenclatura = str(address).split(",")[0].strip().upper()

    head = _ROAD_HEAD_RE.match(nomenclatura)
    if not head:
        return null_vec

    road_type, road_num, suffix = head.groups()
    bis         = bool(suffix) and suffix.upper() == 'BIS'
    road_letter = suffix if (suffix and not bis) else None
    road_code   = _ROAD_CODES.get(road_type.upper(), 0)

    rest = nomenclatura[head.end():]
    cross_num = cross_letter = distance = None

    cross_m = _CROSS_RE.search(rest)
    if cross_m:
        cross_num, cross_letter, distance = cross_m.groups()
        if not distance:
            dist_m = _DIST_QUALIFIER_RE.search(rest)
            if dist_m:
                distance = dist_m.group(1)
    else:
        no_hash_m = _CROSS_NO_HASH_RE.search(rest)
        if no_hash_m:
            cross_num, cross_letter, distance = no_hash_m.groups()

    return np.array([
        road_code * 1000 + _to_decimal(road_num, road_letter, bis),
        _to_decimal(cross_num, cross_letter),
        float(distance) if distance else 0.0,
    ], dtype=np.float32)

def add_handshake(df, source_col="direccion_norm"):
    """Inserts integration_handshake (3D vector as tuple) immediately after source_col."""
    loc = df.columns.get_loc(source_col) + 1
    df.insert(loc, "integration_handshake",
              df[source_col].apply(lambda x: tuple(address_to_vector(x))))
    return df

### 3.3. Parseo de coordenadas a WGS84

In [ ]:
import re
from typing import Optional, Tuple

# ── Empty / non-parseable sentinel values ────────────────────────────────────
_COORD_EMPTY = frozenset({
    '', '-', ' ', 'n/a', 'na', 'nd', 's/d',
    'sin dato', 'ninguno', 'no tengo', 'sin coordenadas', 'no aplica',
})

# ── Unicode quote normalization table ────────────────────────────────────────
# Run FIRST in parse_coords. After this, only ASCII ' and " are in the string,
# so the DMS regex uses plain ' and " without any escaping gymnastics.
_QUOTE_NORM = str.maketrans({
    '‘': "'", '’': "'", 'ʼ': "'", '′': "'",  # smart/prime → '
    '“': '"', '”': '"', 'ʺ': '"', '″': '"',  # smart/double-prime → "
})

# ── Google Maps URL patterns ─────────────────────────────────────────────────
_GMAPS_AT = re.compile(r'@(-?\d{1,3}\.\d+),(-?\d{1,3}\.\d+)')
_GMAPS_Q  = re.compile(r'[?&]q=(-?\d{1,3}\.\d+)[,+](-?\d{1,3}\.\d+)', re.IGNORECASE)
_GMAPS_LL = re.compile(r'[?&]ll=(-?\d{1,3}\.\d+),(-?\d{1,3}\.\d+)', re.IGNORECASE)

# ── Coordinate label stripper ─────────────────────────────────────────────────
# Handles: LATITUD, LATITUD:, LATITUD :, LAT:, LON:, LONGITUD:, etc.
_COORD_LABELS = re.compile(
    r'\b(?:LATITUD|LONGITUD|LAT|LON|LONG)\b\s*:?\s*', re.IGNORECASE
)

# ── DMS regex ─────────────────────────────────────────────────────────────────
# Built with adjacent string literals so ' and " appear as plain ASCII chars,
# avoiding the double-quote-inside-double-quoted-raw-string escaping trap.
# Input must already be normalized (Unicode quotes → ASCII via _normalize).
#
# Matches:  3°27'24.5"N 76°32'35.2"W
#           3°22'22.02"N 76°33'13"O        (O = Oeste = West)
#           3°22'20.67"N 76°33'17.41"W     (after label-stripping)
_DMS = re.compile(
    r'(\d{1,3})\s*[°*]\s*(\d{1,2})\s*'
    "'"
    r'\s*(\d{1,2}(?:[.,]\d+)?)\s*'
    '"'
    r'{0,2}\s*([NSns])'
    r'[\s,;]+'
    r'(\d{1,3})\s*[°*]\s*(\d{1,2})\s*'
    "'"
    r'\s*(\d{1,2}(?:[.,]\d+)?)\s*'
    '"'
    r'{0,2}\s*([EWOewo])',
)

# ── Decimal degrees (dot separator) ─────────────────────────────────────────
_DD = re.compile(r'(-?\d{1,3}\.\d+)\s*[,;\s]\s*(-?\d{1,3}\.\d+)')

# ── MAGNA-SIRGAS planar (EPSG:3115) ──────────────────────────────────────────
_PLANAR_LABELED = re.compile(
    r'(?:[EN][:\s]*)(\d{6,7}(?:[.,]\d+)?)\s*[,;\s]\s*(?:[EN][:\s]*)(\d{6,7}(?:[.,]\d+)?)',
    re.IGNORECASE,
)
_PLANAR_BARE = re.compile(
    r'(?<!\d)(\d{6,7}(?:\.\d+)?)\s*[,;\s]\s*(\d{6,7}(?:\.\d+)?)(?!\d)'
)


def _normalize(s: str) -> str:
    """Normalize Unicode quotes to ASCII and collapse all whitespace to single space."""
    return re.sub(r'\s+', ' ', s.translate(_QUOTE_NORM)).strip()


def _fix_decimal(s: str) -> float:
    return float(str(s).replace(',', '.'))


def _dms_to_dd(deg: str, mins: str, secs: str, hemi: str) -> float:
    dd = float(deg) + float(mins) / 60.0 + _fix_decimal(secs) / 3600.0
    return -dd if hemi.upper() in ('S', 'W', 'O') else dd


def _valid_wgs84(lat: float, lon: float) -> bool:
    return -90.0 <= lat <= 90.0 and -180.0 <= lon <= 180.0


def _plausible_cali(lat: float, lon: float) -> bool:
    """Bounding box for Valle del Cauca and surroundings."""
    return 2.0 <= lat <= 5.5 and -78.0 <= lon <= -75.0


def _try_gmaps_url(s: str) -> Optional[Tuple[float, float]]:
    for pat in (_GMAPS_AT, _GMAPS_Q, _GMAPS_LL):
        m = pat.search(s)
        if m:
            lat, lon = float(m.group(1)), float(m.group(2))
            if _valid_wgs84(lat, lon):
                return lat, lon
    return None


def _try_dms(s: str) -> Optional[Tuple[float, float]]:
    # Strip LATITUD/LONGITUD labels, then collapse remaining whitespace
    clean = re.sub(r'\s+', ' ', _COORD_LABELS.sub(' ', s)).strip()
    m = _DMS.search(clean)
    if not m:
        return None
    lat = _dms_to_dd(m.group(1), m.group(2), m.group(3), m.group(4))
    lon = _dms_to_dd(m.group(5), m.group(6), m.group(7), m.group(8))
    return (lat, lon) if _valid_wgs84(lat, lon) else None


def _try_decimal(s: str) -> Optional[Tuple[float, float]]:
    # First pass: standard dot-decimal  "3.491, -76.519"
    m = _DD.search(s)
    if not m:
        # Second pass: comma-as-decimal-separator  "3,4910915, -76,5191746"
        # Replace digit,digit → digit.digit (field separator ", " is unaffected
        # because it has a space after the comma)
        m = _DD.search(re.sub(r'(\d),(\d)', r'\1.\2', s))
    if not m:
        return None
    a, b = float(m.group(1)), float(m.group(2))
    for lat, lon in ((a, b), (b, a)):
        if _valid_wgs84(lat, lon) and _plausible_cali(lat, lon):
            return lat, lon
    for lat, lon in ((a, b), (b, a)):
        if _valid_wgs84(lat, lon):
            return lat, lon
    return None


def _try_magna_sirgas(s: str) -> Optional[Tuple[float, float]]:
    """MAGNA-SIRGAS / Colombia West (EPSG:3115) → WGS84. No-op if pyproj absent."""
    try:
        from pyproj import Transformer
    except ImportError:
        return None
    m = _PLANAR_LABELED.search(s) or _PLANAR_BARE.search(s)
    if not m:
        return None
    e, n = _fix_decimal(m.group(1)), _fix_decimal(m.group(2))
    in_range_en = 900_000 <= e <= 1_300_000 and 700_000 <= n <= 1_200_000
    in_range_ne = 900_000 <= n <= 1_300_000 and 700_000 <= e <= 1_200_000
    if not (in_range_en or in_range_ne):
        return None
    if in_range_ne and not in_range_en:
        e, n = n, e
    try:
        t = Transformer.from_crs('EPSG:3115', 'EPSG:4326', always_xy=True)
        lon, lat = t.transform(e, n)
        return (lat, lon) if _valid_wgs84(lat, lon) else None
    except Exception:
        return None


def parse_coords(value) -> Optional[Tuple[float, float]]:
    """
    Parses a coordinate string → (lat, lon) WGS84 decimal degrees, or None.

    Pipeline (all formats handled after Unicode normalization):
      1. Google Maps URL   →  /@lat,lon  |  ?q=lat,lon  |  &ll=lat,lon
      2. DMS               →  3°27'24.5"N 76°32'35.2"W  |  3°22'13"O
         DMS labeled       →  LATITUD 3°22'20.67"N LONGITUD 76°33'17.41"W
         DMS label+colon   →  LATITUD: 3°22'20.73"N LONGITUD: 76°33'17.80"W
      3. Decimal dot       →  3.456789, -76.543210
         Decimal comma     →  3,4910915, -76,5191746
      4. MAGNA-SIRGAS / Colombia West (EPSG:3115)  —  requires pyproj
    """
    if value is None:
        return None
    raw = str(value).strip()
    if not raw:
        return None
    s = _normalize(raw)
    if not s or s.lower() in _COORD_EMPTY:
        return None
    return (
        _try_gmaps_url(s)
        or _try_dms(s)
        or _try_decimal(s)
        or _try_magna_sirgas(s)
    )


def coords_to_wgs84(value) -> str:
    """Returns 'lat, lon' in WGS84 decimal, or '' if the value can't be parsed."""
    result = parse_coords(value)
    if result is None:
        return ''
    lat, lon = result
    return f'{lat:.6f}, {lon:.6f}'


### 3.4. Utilidades de matching (canonicalización, geo y validaciones)

Funciones de apoyo para el matching mejorado de la sección 6:

- **`canonicalize_for_match`** — limpieza *solo para comparar* (no altera `direccion_norm`): quita ruido de unidad (`APTO 203B`, `TORRE 85`), colapsa calificadores direccionales (`NORTE` → `N`) y despega tipos de vía pegados al número (`CALLE5B3` → `CL 5B3`).
- **`parse_latlon` / `haversine_m`** — habilitan el nivel geoespacial: distancia en metros entre coordenadas WGS84 ya parseadas en 3.3.
- **`barrio_ok` / `coherent`** — *guards* de precisión: un match por similitud de texto solo se acepta si el barrio concuerda y los números viales de ambas direcciones no se contradicen. Esto evita falsos positivos del estilo `CL 8 # 38-120` ↔ `CL 8 # 39-120` (cuadras distintas con texto casi idéntico).
- **`corner_key`** — huella de esquina independiente del orden: `KR 72 CON CL 11` ≡ `CL 11 # 72-15`.

In [ ]:
import math
from rapidfuzz import fuzz

# ── Matching-only canonicalization (applied on top of normalize_address) ──────
# Unit descriptors locate the dwelling INSIDE the property, not the property
# itself — they only add noise to address comparison.
_UNIT_NOISE_RE = re.compile(
    r'\b(APTO?|APARTAMENTO|APT|TORRE|CASA|BLOQUE|BLQ|BQ|PISO|LOCAL|OFICINA|OFC|'
    r'ETAPA|INTERIOR|INT|UNIDAD|CONJUNTO|EDIFICIO|EDIF|URBANIZACION|URB)\.?\s*[\w-]*',
    re.IGNORECASE,
)

# Directional qualifiers → single letter glued to the number, so that
# "AV 5 NORTE", "AV 5 NTE." and "AV 5N" all canonicalize identically.
_DIRECTIONALS = [
    (re.compile(r'\bNORTE\b|\bNTE\.?\b'), 'N'),
    (re.compile(r'\bOESTE\b|\bOE\.?\b'),  'O'),
    (re.compile(r'\bESTE\b'),             'E'),
    (re.compile(r'\bSUR\b'),              'S'),
]

# Road word glued to its number ("CALLE5B3") — normalize_address misses these
# because all its patterns are word-bounded.
_GLUED_ROAD_RE = re.compile(
    r'\b(CALLE|CARRERA|AVENIDA|DIAGONAL|TRANSVERSAL|CLL|CRA|KRA|CL|KR|AV|DG|TV)(\d)',
    re.IGNORECASE,
)
# Safety net for road words that survived normalize_address (rare glued cases)
_ROAD_WORDS = [
    (re.compile(r'\b(CALLE|CLL)\b'), 'CL'),
    (re.compile(r'\b(CARRERA|CRA|KRA|CR)\b'), 'KR'),
    (re.compile(r'\bAVENIDA\b'), 'AV'),
    (re.compile(r'\bDIAGONAL\b'), 'DG'),
    (re.compile(r'\bTRANSVERSAL\b'), 'TV'),
]

def canonicalize_for_match(addr) -> str:
    """
    Matching-only cleanup applied on top of normalize_address output.
    Keeps only the nomenclature (text before the first comma), removes unit
    noise, and collapses directional qualifiers.

      "KR 60A # 11B-31 APTO 203B, Santa Anita, Cali" → "KR 60A # 11B-31"
      "AV 5 NORTE # 22-40, Versalles, Cali"          → "AV 5N # 22-40"
      "CALLE5B3 # 38-16, San Fernando, Cali"          → "CL 5B3 # 38-16"
    """
    if not addr or str(addr).strip() in {"", "-"}:
        return ""
    s = str(addr).split(",")[0].strip().upper()
    s = _GLUED_ROAD_RE.sub(lambda m: m.group(1) + " " + m.group(2), s)
    for rx, rep in _ROAD_WORDS:
        s = rx.sub(rep, s)
    s = _UNIT_NOISE_RE.sub(" ", s)
    for rx, rep in _DIRECTIONALS:
        s = rx.sub(rep, s)
    s = re.sub(r'(\d)\s+([NOSE])\b', r'\1\2', s)   # glue letter: "5 N" → "5N"
    return re.sub(r'\s+', ' ', s).strip()

# ── Geographic helpers (for the geo matching tier) ───────────────────────────
def parse_latlon(val):
    """'3.4213, -76.5312' → (lat, lon) if inside the Cali bounding box, else None."""
    m = re.match(r'^\s*(-?\d+\.\d+)\s*,\s*(-?\d+\.\d+)\s*$', str(val))
    if not m:
        return None
    lat, lon = float(m.group(1)), float(m.group(2))
    return (lat, lon) if (2.9 <= lat <= 4.1 and -77.0 <= lon <= -76.0) else None

def haversine_m(a, b) -> float:
    """Great-circle distance in meters between two (lat, lon) tuples."""
    la1, lo1, la2, lo2 = map(math.radians, (*a, *b))
    h = math.sin((la2-la1)/2)**2 + math.cos(la1)*math.cos(la2)*math.sin((lo2-lo1)/2)**2
    return 6371000 * 2 * math.asin(math.sqrt(h))

# ── Validation guards (precision protection for the looser tiers) ────────────
def barrio_ok(b1, b2, threshold=75) -> bool:
    """True if either barrio is missing OR both fuzzy-agree (token_sort >= threshold)."""
    b1, b2 = str(b1).strip(), str(b2).strip()
    if b1 in {"", "-", "nan", "None"} or b2 in {"", "-", "nan", "None"}:
        return True
    return fuzz.token_sort_ratio(b1.upper(), b2.upper()) >= threshold

def road_nums(vec):
    """Integer street numbers present in a parsed address vector: {road, cross}."""
    return {n for n in (int(vec[0] % 1000), int(vec[1])) if n != 0}

def coherent(v1, v2, strict=True) -> bool:
    """
    Numeric guard for text-similarity matches: the street numbers of both
    addresses must agree. Without this, fuzzy methods pair "CL 8 # 38-120"
    with "CL 8 # 39-120" (different block) on pure text similarity.

    strict=True  → number sets must be equal or one contained in the other
    strict=False → at least one shared number (used by the geo tier, where
                   physical proximity is already strong evidence)
    If either side is unparseable there is nothing to contradict → passes.
    """
    n1, n2 = road_nums(v1), road_nums(v2)
    if not n1 or not n2:
        return True
    if strict:
        return n1 == n2 or n1 <= n2 or n2 <= n1
    return bool(n1 & n2)

def corner_key(canon, vec):
    """
    Order-independent corner fingerprint: the sorted pair of both street
    numbers. Captures that "KR 72 CON CL 11", "CL 11 # 72-15" and
    "KR 72 # 11-30" all describe the same corner.

      "KR 72 CON CL 11" → (11.0, 72.0)
      "CL 4 # 38E-07"   → (4.0, 38.05)   ≡ "KR 38E # 4-145"
    """
    if not canon or vec[0] == 0:
        return None
    rn = round(float(vec[0]) % 1000, 2)
    cn = round(float(vec[1]), 2)
    if cn == 0:
        # cross street via "CON KR 8" / "X CL 5" / "ESQUINA ..." phrasing
        m = re.search(
            r'\b(?:CON|X|ESQUINA)\s+(?:AU|AV|AC|AK|KR|CL|CV|CQ|CT|DG|TV|TC)?\s*(\d+)\s*([A-Z]?)\b',
            canon,
        )
        if m:
            cn = round(float(m.group(1)) + ((ord(m.group(2)) - 64) * 0.01 if m.group(2) else 0), 2)
    return tuple(sorted((rn, cn))) if cn else None

## 4. Procesamiento de datos "EDAN"

### 4.1. Lectura

In [ ]:
SPREADSHEET_ID = "1QRLezOtMTZpePluDl7VVzxINYLgvACVB76z54vJNpfE"
SHEET_NAME = "EDAN 100826 - Datos Madre"
SERVICE_ACCOUNT_FILE = "service_account.json"

creds = Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE,
    scopes=["https://www.googleapis.com/auth/spreadsheets.readonly"],
)
gc = gspread.authorize(creds)

sheet = gc.open_by_key(SPREADSHEET_ID).worksheet(SHEET_NAME)
data = sheet.get_all_records()
df = pd.DataFrame(data)

print(f"Shape: {df.shape}")
#df.head()

### 4.2. Limpieza de datos

In [ ]:
df = df.drop(columns=["DESCRIPCION CENTRALIZADA", 
                      "REFERENCIADO EN EL MAPA", 
                      "ID",
                      "Direccion",
                      "Color semaforo"
                      ])

In [ ]:
df = df.rename(columns={
    "Prioridad": "prioridad",
    "Estado": "estado",
    "Comuna - Corregimiento": "comuna_corregimiento",
    "Zona": "zona",
    "Barrio": "barrio_vereda",
    "Tipo": "tipo_estructura",
    "Referencia": "punto_referencia",
    "DIRECCION COMPLETA": "direccion",
    "Descripción": "descripcion",
    "Normalizacion": "normalizacion",
    "Personas Evacuadas": "n_personas_evacuadas",
    "Seres sintientes": "n_seres_sintientes",
    "Atencion de la respuesta": "personal_atencion",
    "Colapso Total": "n_colapsados_total",
    "Colapso Parcial": "n_colapsados_parcial",
    "Daños": "n_danos",
    "Atrapamientos": "n_atrapamientos",
    "Rescatados": "n_rescatados",
    "Fallecidos": "n_fallecidos",
    "NOMBRE DELEGADO": "nombre_delegado",
    "OBSERVACIONES (Gestiòn del riesgo)": "observaciones_gred",
    "DESAPARECIDOS": "n_desaparecidos",
    "COORDENADAS": "coords",
    "DESCRIPCION DE LA SITUACION": "descripcion_2",
    "Observaciones generales (FORMULARIO)": "observaciones_generales",
    "NOMBRE LIDER": "nombre_lider",
    "TELEFONO": "num_telefono",
    "GRUPO ": "grupo",
})

In [ ]:
df["prioridad"] = df["prioridad"].str.title()
df["direccion"] = df["direccion"].str.title()
df["nombre_lider"] = df["nombre_lider"].str.title()
df["nombre_delegado"] = df["nombre_delegado"].str.title()

In [ ]:
empty_values = {"", "-", " "}

cols_check = [c for c in df.columns if c != "sitio_id"]

anulados = df["estado"].str.upper() == "ANULADO"
sin_datos = df[cols_check].apply(
    lambda row: all(str(v).strip() in empty_values for v in row), axis=1
)

print(f"Registros anulados:  {anulados.sum()}")
print(f"Registros sin datos: {sin_datos.sum()}")
print(f"Total a eliminar:    {(anulados | sin_datos).sum()}")

df = df[~(anulados | sin_datos)].reset_index(drop=True)
print(f"Registros restantes: {len(df)}")

### 4.3. Asignación de índices artificiales y orden

In [ ]:
# Deterministic sitio_id (seed 42, assigned by row position on first run).
# Once section 6.10 persists the column to the sheet, the sheet values become
# the source of truth: they are kept verbatim and new IDs are generated only
# for rows that do not have one yet. This survives row inserts/reorders.
import re as _re_ids

chars = string.ascii_uppercase + string.digits
# gspread's get_all_records numericises values: "0224" -> 224, "00E9" -> 0.0
# (unrecoverable). New IDs therefore avoid digit-only and dEdd patterns, and
# the persisted column is re-read as raw text below.
_ID_AMBIGUO = _re_ids.compile(r"\d{4}|\d+E\d+")

def generar_ids(seed, n, taken=()):
    """Deterministic 4-char ID sequence, skipping taken and numeric-lookalikes."""
    rng = random.Random(seed)
    seen, out = set(taken), []
    while len(out) < n:
        cand = "".join(rng.choices(chars, k=4))
        if cand not in seen and not _ID_AMBIGUO.fullmatch(cand):
            seen.add(cand)
            out.append(cand)
    return out

if "sitio_id" in df.columns:
    # re-read the ID column as raw text (col_values does not numericise)
    _pos = df.columns.get_loc("sitio_id") + 1
    _raw = sheet.col_values(_pos)[1:]
    _raw = (_raw + [""] * len(df))[:len(df)]
    cur = pd.Series(_raw, index=df.index).astype(str).str.strip()
    nuevos = iter(generar_ids(42, int(cur.eq("").sum()), taken=set(cur[cur.ne("")])))
    df["sitio_id"] = [c if c else next(nuevos) for c in cur]
    df.insert(0, "sitio_id", df.pop("sitio_id"))
    print(f"sitio_id reutilizado de la hoja ({int(cur.ne('').sum())} existentes)")
else:
    df.insert(0, "sitio_id", generar_ids(42, len(df)))
    print(f"sitio_id generado por posicion ({len(df)} nuevos)")
#df.head(10)

In [ ]:
df.shape

In [ ]:
def group_after(cols, anchor, companion):
    cols = [c for c in cols if c != companion]
    cols.insert(cols.index(anchor) + 1, companion)
    return cols

all_cols = df.columns.tolist()
n_cols = [c for c in all_cols if c.startswith("n_")]
anchor = all_cols.index("n_personas_evacuadas")
before = [c for c in all_cols[:anchor] if not c.startswith("n_")]
after = [c for c in all_cols[anchor:] if not c.startswith("n_")]

order = before + n_cols + after
order = group_after(order, "descripcion", "descripcion_2")
df = df[order]

### 4.4. Normalización de direcciones y Vectorización

In [ ]:
df.insert(df.columns.get_loc("direccion") + 1, "direccion_norm", df["direccion"].apply(normalize_address))

In [ ]:
#df['direccion_norm'].head()

In [ ]:
df = add_handshake(df, source_col="direccion_norm")
df[["direccion_norm", "integration_handshake"]].head(10)

In [ ]:
df["coords"] = df["coords"].apply(lambda v: coords_to_wgs84(v) or v)

In [ ]:
df_edan = df

### 4.5. Hallazgos — EDAN

- **2.159 filas crudas → 2.067 sitios útiles**: se eliminan 92 registros con estado `ANULADO`. No se encontraron filas totalmente vacías.
- **Solo ~7 % de los sitios EDAN tiene coordenadas parseables a WGS84** (139 / 2.067 en la corrida de referencia, 13-ago-2026). La columna `coords` mezcla formatos: URL de Google Maps, DMS con comillas Unicode, decimales con coma y MAGNA-SIRGAS planas — de ahí el parser multiformato de 3.3.
- **La digitación de direcciones es muy heterogénea**: esquinas descritas con `CON` (`KR 72 CON CL 11`), placas sin guion (`CL 10 BIS # 70 34`), calificadores en texto (`AV 5 NORTE`), nombres de edificios dentro de la dirección. `normalize_address` unifica tipo de vía y separadores, pero el emparejamiento exacto sigue siendo insuficiente — esto motiva la cascada de la sección 6.
- Una parte de los sitios queda con huella "débil" (sin cruce `#`): son esquinas o referencias abiertas que no pueden distinguir un predio específico dentro de la vía.

## 5. Procesamiento de datos "Visitas"

### 5.1. Lectura

In [ ]:
SPREADSHEET_ID = "1SIzarDbjtaD6JVM7cUHWqrcLBJNgYj6ZooHVU9tRKN4"
SHEET_NAME = "Respuestas de formulario 1"
SERVICE_ACCOUNT_FILE = "service_account.json"

creds = Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE,
    scopes=["https://www.googleapis.com/auth/spreadsheets.readonly"],
)
gc = gspread.authorize(creds)

sheet = gc.open_by_key(SPREADSHEET_ID).worksheet(SHEET_NAME)
data = sheet.get_all_records()
df = pd.DataFrame(data)

print(f"Shape: {df.shape}")

### 5.2. Limpieza de datos

In [ ]:
df = df.drop(columns=["Marque con una X si este formulario fue diligenciado por el grupo de arquitectos e ingenieros voluntarios para la evaluación de edificaciones en riesgo de colapso.\n\nDE LO CONTRATIO DEJA SIN MARCAR LA CASILLA",
                      "Columna 22", 
                      "Columna 1",
                      "Colapso"
                      ])

In [ ]:
df = df.rename(columns={
    "Marca temporal": "timestamp",
    "Dirección de correo electrónico": "email",
    "Dirección completa": "direccion",
    "Coordenadas Geográficas": "coords",
    "Descripción completa de la situación encontrada": "descripcion",
    "Cantidad de personas fallecidos (EN NÚMEROS POR FAVOR)": "n_fallecidos",
    "Cantidad de personas atrapadas  (EN NÚMEROS POR FAVOR)": "n_atrapamientos",
    "Cantidad de personas rescatadas  (EN NÚMEROS POR FAVOR)": "n_rescatados",
    "Suba los soportes que considere necesarios (Vídeos, fotos, audios, documentos)": "evidencia_soporte",
    "Nombre de la persona que diligencia el reporte": "nombre_diligenciador",
    "¿Se necesita evacuación preventiva de las personas?": "evacuacion_preventiva",
    "Cantidad de personas aproximadas que  necesitan evacuar (POR FAVOR DIGITALICE EN NÚMEROS) \n\nSi al momento de la visita, la edificación ya fué evacuada, diligenciar en observaciones y colocar el aproximado de personas que evacuaron (si cuenta con el dato)": "personas_necesitan_evacuar",
    "Cantidad de Casas, vivienda o apartamentos caracterizados y atendidos  (POR FAVOR DIGITALICE EN NÚMEROS) ": "n_estructuras_caracterizadas",
    "Nombre del Edificio, Centro Comercial, Hospital, Unidad o Conjunto Residencial (SI APLICA, en caso contrario diligenciar como NA)": "nombre_estructura",
    "Nombre del organismo que pertenece. \n\nEn caso de ser parte del grupo de ingenieros y arquitectos voluntarios ir a la siguiente pregunta": "nombre_organismo",
    "Nivel de riesgo ": "nivel_riesgo",
    "Consecutivo - id (solo si fue asignado)": "consecutivo_gred",
    "¿La edificación requiere demolición?": "requiere_demolicion",
    "Describa las observaciones que considere pertinentes a la pregunta anterior": "observaciones",
    "Requieren evacuación": "requieren_evacuacion",
    "Observaciones generales":"observaciones_generales",
    "Comuna": "comuna_corregimiento",
    "Barrio": "barrio_vereda",
    "Tipo de edificación atendida": "tipo_estructura",
    "Situación de la vivienda": "estado_estructura"
    
})

In [ ]:
df.columns

In [ ]:
df["barrio_vereda"] = df["barrio_vereda"].str.title()
df["comuna_corregimiento"] = df["comuna_corregimiento"].str.title()
df["tipo_estructura"] = df["tipo_estructura"].str.title()
df["nombre_diligenciador"] = df["nombre_diligenciador"].str.title()
df["nombre_estructura"] = df["nombre_estructura"].str.title()
df["nombre_diligenciador"] = df["nombre_diligenciador"].str.title()
df["estado_estructura"] = df["estado_estructura"].str.title()
df["requiere_demolicion"] = df["requiere_demolicion"].str.title()
df["direccion"] = df["direccion"].str.title()

In [ ]:
df.head()

### 5.3. Asignación de índices artificiales y orden

In [ ]:
# Deterministic visita_id (seed 88). Same contract as sitio_id: sheet
# values (once persisted by section 6.10) are the source of truth; new IDs
# only for rows without one. Column re-read as raw text (no numericising).
if "visita_id" in df.columns:
    _pos = df.columns.get_loc("visita_id") + 1
    _raw = sheet.col_values(_pos)[1:]
    _raw = (_raw + [""] * len(df))[:len(df)]
    cur = pd.Series(_raw, index=df.index).astype(str).str.strip()
    nuevos = iter(generar_ids(88, int(cur.eq("").sum()), taken=set(cur[cur.ne("")])))
    df["visita_id"] = [c if c else next(nuevos) for c in cur]
    df.insert(0, "visita_id", df.pop("visita_id"))
    print(f"visita_id reutilizado de la hoja ({int(cur.ne('').sum())} existentes)")
else:
    df.insert(0, "visita_id", generar_ids(88, len(df)))
    print(f"visita_id generado por posicion ({len(df)} nuevos)")
df.head()

### 5.4. Normalización de direcciones y Vectorización

In [ ]:
df.insert(df.columns.get_loc("direccion") + 1, "direccion_norm", df["direccion"].apply(normalize_address))

In [ ]:
_empty = {"", "-", " "}

def _with_location(row):
    norm = str(row["direccion_norm"]).strip()
    if norm in _empty:
        return norm
    barrio = str(row["barrio_vereda"]).strip()
    if barrio and barrio not in _empty:
        return f"{norm}, {barrio}, Cali"
    return f"{norm}, Cali"

df["direccion_norm"] = df.apply(_with_location, axis=1)

In [ ]:
df = add_handshake(df, source_col="direccion_norm")
df[["direccion_norm", "integration_handshake"]].head(10)

In [ ]:
df["coords"] = df["coords"].apply(lambda v: coords_to_wgs84(v) or v)

In [ ]:
df_visitas = df

### 5.5. Hallazgos — Visitas

- **~1.040 respuestas de formulario** (el sheet está vivo: crece entre corridas). No hay filas anuladas; todas se conservan.
- **~32 % de las visitas tiene coordenadas parseables** (334 / 1.041 en la corrida de referencia) — cobertura mucho mayor que en EDAN, lo que convierte a las coordenadas en una señal valiosa de emparejamiento que el matching por texto no aprovechaba.
- Las direcciones de formulario traen **ruido de unidad** (`APTO 203B`, `TORRE 85`, `PISO 2`) y **nombres de conjuntos** dentro del campo dirección; por eso a `direccion_norm` se le añade `, {barrio}, Cali` (desambiguación para similitud de texto) y la comparación usa `canonicalize_for_match` (3.4).
- Varias visitas reportan el **mismo predio con placas distintas** (`KR 57 # 3-15` vs `KR 57 # 3-88`): son entradas o torres distintas del mismo frente de cuadra — un match exacto jamás las une.

## 6. Integración

### 6.1. Método de matching — baseline (v1)

Un join exacto por `direccion_norm` falla entre fuentes con digitación tan heterogénea. El emparejamiento baseline se hace en 3 niveles — el primero que acierta gana:

1. **`handshake`** — igualdad exacta de la huella canónica (`make_handshake`): mismo tipo de vía, número, cruce y placa. Solo se usan huellas *fuertes* (que contienen `#`); una huella como `KR72` sin cruce emparejaría cualquier predio de esa carrera.
2. **`vector`** — distancia euclidiana entre los vectores 3D de `integration_handshake` ≤ `vector_tol`. Tolera diferencias a nivel de letra/BIS (sub-cuadra). Igual que arriba, requiere cruce (`vector[1] != 0`) en ambos lados.
3. **`fuzzy`** — similitud de texto (`rapidfuzz` `token_sort_ratio`) sobre `direccion_norm` completa (incluye barrio) ≥ `fuzzy_threshold`. Es el fallback para direcciones que el parser no pudo vectorizar (vector nulo o sin cruce).

Cada visita se empareja con **máximo un** sitio EDAN. Varias visitas pueden apuntar al mismo sitio (visitas repetidas al mismo predio), lo cual es correcto operativamente.

**Hallazgo — por qué el baseline deja ~500 visitas sin match:**

- `vector_tol = 0.05` solo tolera diferencias de letra en la sub-cuadra; **cualquier diferencia de placa rompe el match** (`KR 57 # 3-15` vs `KR 57 # 3-88` distan 73 unidades).
- `fuzzy_threshold = 92` es casi un join exacto: solo aporta ~40 matches.
- **Las coordenadas no se usan** a pesar de que ~32 % de visitas y ~7 % de sitios EDAN las tienen en WGS84.
- Las **esquinas** se describen en órdenes distintos (`KR 72 CON CL 11` vs `CL 11 # 72-15`) y nunca emparejan.
- Calificadores direccionales (`AV 5 NORTE` vs `AV 5N`) y ruido de unidad (`APTO`, `TORRE`) contaminan tanto la huella como la similitud de texto.

In [ ]:
from rapidfuzz import process, fuzz

def build_match_table(df_edan, df_visitas, vector_tol=0.05, fuzzy_threshold=92.0):
    """
    Matches each visita to at most one EDAN site using direccion_norm.

    Tiers (first hit wins):
      1. handshake — exact canonical fingerprint equality (make_handshake).
                     Only strong fingerprints (containing '#') are used: a key
                     like "KR72" would match any site on that road.
      2. vector    — Euclidean distance between integration_handshake vectors
                     <= vector_tol. Requires cross info (vector[1] != 0) on
                     both sides for the same reason.
      3. fuzzy     — token_sort_ratio on full direccion_norm (includes barrio)
                     >= fuzzy_threshold. Fallback for weak/unparseable addresses.

    match_score semantics per tier:
      handshake → 100.0 | vector → distance (lower is better) | fuzzy → ratio 0-100

    Returns one row per visita: visita_id | sitio_id | match_method | match_score
    """
    # Tier 1 index: strong handshake -> first sitio_id
    hs_index = {}
    for sid, key in zip(df_edan["sitio_id"], df_edan["direccion_norm"].apply(make_handshake)):
        if key and "#" in key and key not in hs_index:
            hs_index[key] = sid

    # Tier 2 index: vectors with cross info
    vecs = np.array([list(v) for v in df_edan["integration_handshake"]], dtype=np.float64)
    has_cross = vecs[:, 1] != 0.0
    edan_vecs = vecs[has_cross]
    edan_vec_sids = df_edan["sitio_id"].to_numpy()[has_cross]

    # Tier 3 index: non-empty normalized addresses
    has_text = df_edan["direccion_norm"].astype(str).str.strip().ne("")
    edan_texts = df_edan.loc[has_text, "direccion_norm"].tolist()
    edan_text_sids = df_edan.loc[has_text, "sitio_id"].tolist()

    rows = []
    for _, visita in df_visitas.iterrows():
        addr = str(visita["direccion_norm"]).strip()
        sid = method = score = None

        key = make_handshake(addr) if addr else ""
        if key and "#" in key and key in hs_index:
            sid, method, score = hs_index[key], "handshake", 100.0

        if sid is None:
            v = np.asarray(visita["integration_handshake"], dtype=np.float64)
            if v[1] != 0.0 and len(edan_vecs):
                dists = np.linalg.norm(edan_vecs - v, axis=1)
                best = int(np.argmin(dists))
                if dists[best] <= vector_tol:
                    sid, method = edan_vec_sids[best], "vector"
                    score = round(float(dists[best]), 4)

        if sid is None and addr and edan_texts:
            hit = process.extractOne(addr, edan_texts,
                                     scorer=fuzz.token_sort_ratio,
                                     score_cutoff=fuzzy_threshold)
            if hit:
                sid, method, score = edan_text_sids[hit[2]], "fuzzy", round(hit[1], 1)

        rows.append({"visita_id": visita["visita_id"], "sitio_id": sid,
                     "match_method": method, "match_score": score})

    return pd.DataFrame(rows)

In [ ]:
# Baseline run (kept for comparison against the improved matcher below)
match_table_v1 = build_match_table(df_edan, df_visitas)
print(f"v1: {match_table_v1['sitio_id'].notna().sum()} / {len(match_table_v1)} visitas emparejadas")
match_table_v1["match_method"].value_counts(dropna=False)

### 6.2. Matching mejorado (v2) — cascada de 7 niveles

Se probaron varios enfoques (relajar umbrales del baseline, TF-IDF + coseno, `token_set_ratio`, matching geoespacial) y el mejor resultado lo da una **cascada por confianza**: los métodos exactos deciden primero y los métodos difusos solo actúan sobre lo que queda, siempre con *guards* de validación para no comprar recall con falsos positivos.

| # | Método | Idea | Guard |
|---|--------|------|-------|
| 1 | `handshake` | huella canónica exacta sobre dirección canonicalizada | — (exacto) |
| 2 | `vector` | distancia 3D ≤ 0.05 (sub-cuadra) | — (exacto) |
| 3 | `vector_block` | misma vía y cruce, placa ≤ 40 (mismo frente de cuadra) | barrio |
| 4 | `corner` | huella de esquina independiente del orden | barrio |
| 5 | `geo` | haversine ≤ 40 m (o ≤ 90 m si el sitio EDAN no tiene dirección parseable) | coherencia numérica laxa |
| 6 | `tfidf` | **ML**: TF-IDF de n-gramas de caracteres (2-4) + similitud coseno ≥ 0.82 | coherencia estricta + barrio |
| 7 | `fuzzy` | `token_set_ratio` ≥ 88 (umbral relajado vs 92 del baseline) | coherencia estricta + barrio + largo mínimo |

**Hallazgos del proceso:**

- Los *guards* importan más que los umbrales: relajar el fuzzy de 92 → 88 **sin** coherencia numérica producía matches falsos entre cuadras vecinas (`# 38-120` ↔ `# 39-120`); con el guard, el umbral relajado es seguro.
- `vector_block` y `corner` son los niveles nuevos que más aportan (~65 y ~60 matches): capturan exactamente los patrones de digitación que el baseline no podía unir.
- `geo` aporta pocos matches pero valiosos: incluye sitios EDAN **sin dirección utilizable**, imposibles de emparejar por texto.
- Las visitas que quedan sin match en su mayoría reportan predios que **no existen en el registro EDAN** — no es un fallo del matcher sino cobertura real de las fuentes.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def build_match_table_v2(df_edan, df_visitas,
                         vector_tol=0.05, block_placa_tol=40.0,
                         geo_near_m=40.0, geo_far_m=90.0,
                         tfidf_min=0.82, fuzzy_threshold=88.0,
                         fuzzy_min_len=12):
    """
    Improved matcher: 7-tier cascade ordered by confidence, first hit wins.
    All comparisons run on canonicalize_for_match(direccion_norm).

    Tiers:
      1. handshake    — exact canonical fingerprint (same as v1, but on the
                        canonicalized address: unit noise and directional
                        qualifiers no longer break equality).
      2. vector       — 3D Euclidean distance <= vector_tol (same as v1).
      3. vector_block — same road AND same cross street (dims 0-1 equal),
                        placa within block_placa_tol meters-of-numbering:
                        different entrances/towers of the same block face.
                        Guarded by barrio agreement.
      4. corner       — order-independent corner fingerprint equality
                        ("KR 72 CON CL 11" == "CL 11 # 72-15"), barrio-guarded.
      5. geo          — haversine distance between WGS84 coords:
                        <= geo_near_m always; up to geo_far_m only when the
                        EDAN address is unparseable (coords are then the only
                        evidence). Guarded by loose numeric coherence.
      6. tfidf        — ML text similarity: char n-gram (2-4) TF-IDF + cosine
                        over address + barrio, top-5 candidates >= tfidf_min,
                        guarded by strict numeric coherence + barrio.
      7. fuzzy        — rapidfuzz token_set_ratio >= fuzzy_threshold with the
                        same strict guards; only for addresses long enough to
                        be specific (>= fuzzy_min_len chars).

    match_score semantics: handshake/corner → 100 | vector → distance |
    vector_block → placa delta | geo → meters | tfidf → cosine | fuzzy → ratio.

    Returns one row per visita: visita_id | sitio_id | match_method | match_score
    """
    E = df_edan.reset_index(drop=True)
    V = df_visitas.reset_index(drop=True)

    # Canonical text + parsed vectors on the matching-only representation
    e_canon = E["direccion_norm"].apply(canonicalize_for_match)
    v_canon = V["direccion_norm"].apply(canonicalize_for_match)
    e_vecs = np.array([address_to_vector(a) for a in e_canon], dtype=np.float64)
    v_vecs = np.array([address_to_vector(a) for a in v_canon], dtype=np.float64)
    e_sids = E["sitio_id"].to_numpy()
    e_barrio = E["barrio_vereda"].astype(str).to_numpy()
    v_barrio = V["barrio_vereda"].astype(str).to_numpy()

    # Tier 1 index: strong canonical fingerprints only (must contain '#')
    hs_index = {}
    for i, a in enumerate(e_canon):
        k = make_handshake(a)
        if k and "#" in k and k not in hs_index:
            hs_index[k] = i

    # Tier 2/3 pool: EDAN vectors that carry cross-street info
    has_cross = e_vecs[:, 1] != 0.0

    # Tier 4 index: corner fingerprint -> EDAN row indices
    corner_index = {}
    for i in range(len(E)):
        ck = corner_key(e_canon.iloc[i], e_vecs[i])
        if ck:
            corner_index.setdefault(ck, []).append(i)

    # Tier 5 pool: parseable WGS84 coordinates
    e_geo = [parse_latlon(x) for x in E["coords"]]
    v_geo = [parse_latlon(x) for x in V["coords"]]
    e_geo_idx = [i for i, g in enumerate(e_geo) if g]

    # Tier 6/7 pool: canonical address + barrio as similarity text
    def txt(canon, barrio):
        b = str(barrio).strip()
        return (canon + " " + b.upper()) if b not in {"", "-", "nan"} else canon
    e_text_idx = [i for i in range(len(E)) if e_canon.iloc[i].strip()]
    e_texts = [txt(e_canon.iloc[i], e_barrio[i]) for i in e_text_idx]
    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4))
    Xe = vectorizer.fit_transform(e_texts)

    rows = []
    for j in range(len(V)):
        addr, vv = v_canon.iloc[j], v_vecs[j]
        sid = method = score = None

        # 1. handshake
        if addr:
            k = make_handshake(addr)
            if k and "#" in k and k in hs_index:
                sid, method, score = e_sids[hs_index[k]], "handshake", 100.0

        # 2. vector (exact sub-block)
        if sid is None and vv[1] != 0.0:
            pool = np.where(has_cross)[0]
            d = np.linalg.norm(e_vecs[pool] - vv, axis=1)
            b = int(np.argmin(d))
            if d[b] <= vector_tol:
                sid, method, score = e_sids[pool[b]], "vector", round(float(d[b]), 4)

        # 3. vector_block (same block face, different placa)
        if sid is None and vv[1] != 0.0:
            pool = np.where(has_cross)[0]
            d01 = np.linalg.norm(e_vecs[pool][:, :2] - vv[:2], axis=1)
            cand = pool[d01 <= 0.001]
            if len(cand):
                placa_d = np.abs(e_vecs[cand][:, 2] - vv[2])
                for oi in np.argsort(placa_d):
                    i = cand[oi]
                    if placa_d[oi] <= block_placa_tol and barrio_ok(v_barrio[j], e_barrio[i]):
                        sid, method, score = e_sids[i], "vector_block", round(float(placa_d[oi]), 1)
                        break

        # 4. corner
        if sid is None and addr:
            ck = corner_key(addr, vv)
            if ck and ck in corner_index:
                for i in corner_index[ck]:
                    if barrio_ok(v_barrio[j], e_barrio[i]):
                        sid, method, score = e_sids[i], "corner", 100.0
                        break

        # 5. geo
        if sid is None and v_geo[j] and e_geo_idx:
            dists = sorted((haversine_m(v_geo[j], e_geo[i]), i) for i in e_geo_idx)
            for dm, i in dists[:3]:
                edan_unparseable = e_vecs[i][0] == 0
                limit = geo_far_m if edan_unparseable else geo_near_m
                if dm <= limit and coherent(vv, e_vecs[i], strict=False):
                    sid, method, score = e_sids[i], "geo", round(dm, 1)
                    break

        # 6. tfidf (ML text similarity, guarded)
        if sid is None and addr:
            q = vectorizer.transform([txt(addr, v_barrio[j])])
            sims = cosine_similarity(q, Xe).ravel()
            for b in np.argsort(-sims)[:5]:
                if sims[b] < tfidf_min:
                    break
                i = e_text_idx[int(b)]
                if coherent(vv, e_vecs[i], strict=True) and barrio_ok(v_barrio[j], e_barrio[i]):
                    sid, method, score = e_sids[i], "tfidf", round(float(sims[b]), 3)
                    break

        # 7. fuzzy (guarded, only for specific-enough addresses)
        if sid is None and addr and len(addr) >= fuzzy_min_len:
            hits = process.extract(addr, e_texts, scorer=fuzz.token_set_ratio,
                                   score_cutoff=fuzzy_threshold, limit=5)
            for _, sc, hi in hits:
                i = e_text_idx[hi]
                if coherent(vv, e_vecs[i], strict=True) and barrio_ok(v_barrio[j], e_barrio[i]):
                    sid, method, score = e_sids[i], "fuzzy", round(sc, 1)
                    break

        rows.append({"visita_id": V["visita_id"].iloc[j], "sitio_id": sid,
                     "match_method": method, "match_score": score})

    return pd.DataFrame(rows)

In [ ]:
match_table_v2 = build_match_table_v2(df_edan, df_visitas)

print(f"v1 (baseline): {match_table_v1['sitio_id'].notna().sum()} / {len(match_table_v1)} visitas emparejadas")
print(f"v2 (mejorado): {match_table_v2['sitio_id'].notna().sum()} / {len(match_table_v2)} visitas emparejadas")

# Side-by-side method breakdown
comparison = pd.DataFrame({
    "v1": match_table_v1["match_method"].value_counts(dropna=False),
    "v2": match_table_v2["match_method"].value_counts(dropna=False),
}).fillna(0).astype(int)

# Downstream integration uses the improved table
match_table = match_table_v2
comparison

### 6.3. Nivel semántico con modelo de lenguaje: `match_table_v3`

Modelo **`potion-multilingual-128M`** (model2vec): embeddings estáticos destilados de un transformer multilingüe — corre en CPU, sin torch. Cada registro se codifica como texto semántico: dirección + barrio + nombre de estructura / punto de referencia.

**Interpretación de la dirección antes de codificar (`interpret_address`).** El modelo fue entrenado con lenguaje natural, no con nomenclatura catastral. Se prueba re-escribir `direccion_norm` a texto natural antes de embeber: `CL 5 # 43A-15 N` → `CALLE 5 NUMERO 43A 15 NORTE`. Resultado del A/B sobre los pares ya confirmados (handshake + vector como ground truth):

| Variante | % pares verdaderos con coseno ≥ 0.80 | Matches nuevos (≥ 0.80, con guard) |
|---|---|---|
| Dirección catastral cruda | 41 % | 21 |
| Interpretada a lenguaje natural | **61 %** | 28 (limpios) |

La interpretación acerca los pares verdaderos al umbral (rescata ordinales como `AV 4TA NORTE` ≡ `AV 4 NORTE`, direccionales pegados como `CL 48NTE` ≡ `CL 48 NORTE`), pero también sube la similitud de los negativos — por eso el guard se refuerza:

**`embedding_guard` reforzado.** El LM no entiende numeración catastral (puntúa `CL 16N` vs `CL 17N` con coseno alto siendo calles distintas). Si ambas direcciones parsean: tipo de vía igual + números compatibles por letra + placa dentro de tolerancia. Si alguna NO parsea, ya no hay bypass ciego: se exige **coherencia numérica textual** (los números extraídos por regex de ambos textos deben compartir al menos 2, o todos si hay menos). Solo los registros puramente nominales (sin ningún número) se deciden por similitud + barrio.

Umbral de aceptación: coseno ≥ **0.75** + guard + acuerdo de barrio. La banda [0.75, 0.80) se auditó manualmente: con el guard reforzado sus matches son limpios (p. ej. `CALLE5B3 # 38-16` ≡ `CL 5 B 3 # 38 16`) — el guard permite bajar el umbral sin comprar falsos positivos. La confiabilidad final la decide de todas formas el corte de trust de 6.4.


In [ ]:
from model2vec import StaticModel

# Multilingual static embedding model (distilled from a transformer LM;
# CPU-only, no torch). Encodes each record as a short semantic text so that
# building/complex names contribute to the match, not just the address.
_EMB_EMPTY = {"", "-", "nan", "None", "Na", "NA"}

# The LM was trained on natural language, not cadastral codes. Rewriting
# "CL 5 # 43A-15 N" as "CALLE 5 NUMERO 43A 15 NORTE" before encoding raises
# true-pair cosine >= 0.80 coverage from 41% to 61% on the handshake/vector
# ground truth (A/B tested).
_INTERP_ROAD = {
    "CL": "CALLE", "KR": "CARRERA", "AV": "AVENIDA", "AC": "AVENIDA CALLE",
    "AK": "AVENIDA CARRERA", "DG": "DIAGONAL", "TV": "TRANSVERSAL",
    "CQ": "CIRCUNVALAR", "AUT": "AUTOPISTA",
}
_INTERP_DIR = {"N": "NORTE", "S": "SUR", "E": "ESTE", "W": "OESTE", "O": "OESTE"}
_GLUED_DIR_RE = re.compile(r"(\d+[A-Z]?)([NSEWO])$")

def interpret_address(canon) -> str:
    """Rewrite a canonical cadastral address as natural Spanish for the LM."""
    s = str(canon).strip()
    if not s:
        return s
    toks = s.replace("#", " NUMERO ").replace("-", " ").split()
    out = []
    for t in toks:
        if t in _INTERP_ROAD:
            out.append(_INTERP_ROAD[t])
        elif t in _INTERP_DIR:
            out.append(_INTERP_DIR[t])
        else:
            m = _GLUED_DIR_RE.fullmatch(t)
            if m and m.group(2) in _INTERP_DIR:
                out.append(m.group(1))
                out.append(_INTERP_DIR[m.group(2)])
            else:
                out.append(t)
    return " ".join(out)

def _rec_text(canon, barrio, extra):
    parts = [interpret_address(canon)]
    b = str(barrio).strip()
    if b not in _EMB_EMPTY:
        parts.append(b.title())
    x = str(extra).strip()
    if x not in _EMB_EMPTY:
        parts.append(x[:80])
    return ", ".join(p for p in parts if p)

embedding_model = StaticModel.from_pretrained("minishlab/potion-multilingual-128M")

_E = df_edan.reset_index(drop=True)
_V = df_visitas.reset_index(drop=True)
_sid_pos = {s: i for i, s in enumerate(_E["sitio_id"])}
_vid_pos = {v: i for i, v in enumerate(_V["visita_id"])}

_e_canon = _E["direccion_norm"].apply(canonicalize_for_match)
_v_canon = _V["direccion_norm"].apply(canonicalize_for_match)
_e_addr_vecs = np.array([address_to_vector(a) for a in _e_canon], dtype=np.float64)
_v_addr_vecs = np.array([address_to_vector(a) for a in _v_canon], dtype=np.float64)

_e_rec_texts = [_rec_text(_e_canon.iloc[i], _E["barrio_vereda"].iloc[i], _E["punto_referencia"].iloc[i])
                for i in range(len(_E))]
_v_rec_texts = [_rec_text(_v_canon.iloc[i], _V["barrio_vereda"].iloc[i], _V["nombre_estructura"].iloc[i])
                for i in range(len(_V))]

def _normalize_rows(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)

E_emb = _normalize_rows(embedding_model.encode(_e_rec_texts))
V_emb = _normalize_rows(embedding_model.encode(_v_rec_texts))

def _part_compat(a, b, tol=0.001):
    """Same street number allowing one letterless side: 62A~62 ok, 62A~62B not."""
    if abs(a - b) <= tol:
        return True
    ia, ib = int(a), int(b)
    if ia != ib:
        return False
    return round(a - ia, 2) == 0 or round(b - ib, 2) == 0

def _text_nums(canon):
    """All standalone integers in a canonical address (placa digits included)."""
    return {int(n) for n in re.findall(r"\d+", str(canon))}

def embedding_guard(v, e, canon_v, canon_e, placa_tol=40.0):
    """
    Street-level compatibility for embedding matches. The LM captures textual
    semantics but NOT cadastral numbering (it scores "CL 16N" vs "CL 17N" high
    despite being different streets), so:
    - both addresses parsed -> same road type, letter-compatible road number
      and cross, placa within placa_tol;
    - a side fails to parse -> no blind bypass: the integers extracted from
      both raw texts must share at least 2 (or all, if fewer available);
    - purely name-based records (no numbers) -> similarity + barrio decide.
    """
    v_parsed = v[0] != 0 and v[1] != 0
    e_parsed = e[0] != 0 and e[1] != 0
    if v_parsed and e_parsed:
        if int(v[0] // 1000) != int(e[0] // 1000):
            return False
        if not _part_compat(v[0] % 1000, e[0] % 1000):
            return False
        if not _part_compat(v[1], e[1]):
            return False
        if v[2] and e[2] and abs(v[2] - e[2]) > placa_tol:
            return False
        return True
    n1, n2 = _text_nums(canon_v), _text_nums(canon_e)
    if n1 and n2:
        return len(n1 & n2) >= min(2, len(n1), len(n2))
    return True

def add_embedding_matches(match_table, min_sim=0.75, top_k=5):
    """
    Tier 8 (v3): LM cosine similarity over the still-unmatched visitas.
    Accepts the best candidate >= min_sim that passes embedding_guard and
    barrio agreement. Returns a new match table; existing matches untouched.
    """
    out = match_table.copy().set_index("visita_id")
    unmatched = list(out.index[out["sitio_id"].isna()])
    if not unmatched:
        return out.reset_index()
    rows_pos = [_vid_pos[v] for v in unmatched]
    S = V_emb[rows_pos] @ E_emb.T
    added = 0
    for k, j in enumerate(rows_pos):
        has_text = bool(_v_canon.iloc[j].strip()) or \
                   str(_V["nombre_estructura"].iloc[j]).strip() not in _EMB_EMPTY
        if not has_text:
            continue
        for i in np.argsort(-S[k])[:top_k]:
            sim = float(S[k][i])
            if sim < min_sim:
                break
            if (embedding_guard(_v_addr_vecs[j], _e_addr_vecs[int(i)],
                                _v_canon.iloc[j], _e_canon.iloc[int(i)])
                    and barrio_ok(_V["barrio_vereda"].iloc[j], _E["barrio_vereda"].iloc[int(i)])):
                out.loc[unmatched[k], ["sitio_id", "match_method", "match_score"]] = (
                    _E["sitio_id"].iloc[int(i)], "embedding", round(sim, 3))
                added += 1
                break
    print(f"embedding tier: +{added} matches")
    return out.reset_index()

match_table_v3 = add_embedding_matches(match_table_v2)
match_table = match_table_v3   # downstream integration uses the final table

print(f"v3 (semantico): {match_table_v3['sitio_id'].notna().sum()} / {len(match_table_v3)} visitas emparejadas")
match_table_v3["match_method"].value_counts(dropna=False)


### 6.4. `trust_score` de los emparejamientos + corte de confiabilidad

Cada match recibe un puntaje de confianza **[0, 1]** con dos componentes:

1. **Base por método** — qué tan determinista es la evidencia que lo produjo:

| Método | Base | Método | Base |
|---|---|---|---|
| `handshake` | 0.95 | `geo` | 0.78 |
| `vector` | 0.93 | `tfidf` | 0.75 |
| `vector_block` | 0.82 | `embedding` | 0.72 |
| `corner` | 0.82 | `fuzzy` | 0.70 |

2. **Corroboración por señales independientes** (cada una solo aplica si ambos lados tienen el dato):
   - **Verificación semántica (LM)**: similitud coseno de los embeddings del registro completo — ≥ 0.85 → +0.05 · ≥ 0.70 → +0.02 · < 0.45 → **−0.10**. El modelo de lenguaje audita *todos* los matches, incluso los exactos.
   - **Barrio**: concuerda → +0.03 · contradice → −0.08.
   - **Geografía**: coords a ≤ 60 m → +0.05 · > 250 m → **−0.12** (señal fuerte de falso positivo).

**Corte de confiabilidad (`TRUST_MIN = 0.70`).** Solo se aceptan los matches con trust ≥ 0.70; el resto se rechaza y esas visitas vuelven a quedar sin emparejar (aparecen como `solo_visita` en la integración). Un match cuya evidencia se contradice (barrio distinto, coordenadas a > 250 m o incoherencia semántica según el LM) cae por debajo del corte aunque su método base fuera bueno. El subconjunto de alta confianza (> 0.7) se materializa en 6.8 como `df_integrado_confiable`.


In [ ]:
# trust_score for matched pairs: base confidence by method, adjusted by three
# independent corroboration signals. Clipped to [0.05, 0.99].
METHOD_TRUST_BASE = {
    "handshake": 0.95, "vector": 0.93, "vector_block": 0.82, "corner": 0.82,
    "geo": 0.78, "tfidf": 0.75, "embedding": 0.72, "fuzzy": 0.70,
}

_e_geo_pts = [parse_latlon(x) for x in _E["coords"]]
_v_geo_pts = [parse_latlon(x) for x in _V["coords"]]

def match_trust(visita_id, sitio_id, method) -> float:
    j, i = _vid_pos[visita_id], _sid_pos[sitio_id]
    t = METHOD_TRUST_BASE[method]

    # 1) semantic verification: LM embedding similarity of the full records.
    #    Audits every match, including the exact tiers.
    sim = float(V_emb[j] @ E_emb[i])
    if sim >= 0.85:
        t += 0.05
    elif sim >= 0.70:
        t += 0.02
    elif sim < 0.45:
        t -= 0.10

    # 2) barrio corroboration (only when both sides report one)
    b1 = str(_V["barrio_vereda"].iloc[j]).strip()
    b2 = str(_E["barrio_vereda"].iloc[i]).strip()
    if b1 not in _EMB_EMPTY and b2 not in _EMB_EMPTY:
        t += 0.03 if fuzz.token_sort_ratio(b1.upper(), b2.upper()) >= 75 else -0.08

    # 3) geographic corroboration (only when both sides have coordinates)
    if _v_geo_pts[j] and _e_geo_pts[i]:
        dm = haversine_m(_v_geo_pts[j], _e_geo_pts[i])
        if dm <= 60:
            t += 0.05
        elif dm > 250:
            t -= 0.12

    return round(min(max(t, 0.05), 0.99), 2)

match_table["trust"] = [
    match_trust(r["visita_id"], r["sitio_id"], r["match_method"])
    if pd.notna(r["sitio_id"]) else None
    for _, r in match_table.iterrows()
]

_m = match_table[match_table["sitio_id"].notna()]
print(f"antes del corte: {len(_m)} matches")
print(_m.groupby("match_method")["trust"]
        .agg(["count", "mean", "min", "median", "max"]).round(2).to_string())

# Reliability cutoff: only matches with trust >= TRUST_MIN survive.
# Rejected pairs revert to unmatched (they will appear as solo_visita /
# solo_edan rows in the integration).
TRUST_MIN = 0.70
_reject = match_table["trust"].notna() & (match_table["trust"] < TRUST_MIN)
print(f"\nrechazados por trust < {TRUST_MIN}: {int(_reject.sum())}")
if _reject.any():
    print(match_table.loc[_reject].groupby("match_method")["trust"]
          .agg(["count", "mean"]).round(2).to_string())
match_table.loc[_reject, ["sitio_id", "match_method", "match_score", "trust"]] = \
    [None, None, None, None]

_m = match_table[match_table["sitio_id"].notna()]
print(f"\ndespues del corte: {len(_m)} / {len(match_table)} visitas emparejadas "
      f"(todas con trust >= {TRUST_MIN})")
print(f"sitios EDAN con visita: {_m['sitio_id'].nunique()} / {len(_E)}")
print(_m.groupby("match_method")["trust"]
        .agg(["count", "mean", "min", "median", "max"]).round(2).to_string())


### 6.5. Mezcla horizontal e índice combinado

Merge externo (`outer`) para no perder registros de ninguna fuente de verdad:

- **Sitio EDAN con visita** → fila con ambas fuentes; las columnas repetidas quedan con sufijo `_edan` / `_visita`.
- **Sitio EDAN sin visita** → columnas de visita vacías.
- **Visita sin sitio EDAN** → columnas de EDAN vacías.

El índice `registro_id` combina `sitio_id`-`visita_id`; el lado faltante se marca con `----` (imposible que colisione con un id real, que es alfanumérico).

In [ ]:
df_master = (
    df_edan
    .merge(match_table, on="sitio_id", how="outer")
    .merge(df_visitas, on="visita_id", how="outer", suffixes=("_edan", "_visita"))
)

# Combined index: sitio_id-visita_id, "----" marks the missing side
registro_id = df_master["sitio_id"].fillna("----") + "-" + df_master["visita_id"].fillna("----")
df_master.insert(0, "registro_id", registro_id)
df_master = df_master.set_index("registro_id")

print(f"EDAN: {len(df_edan)} | Visitas: {len(df_visitas)} | Master: {len(df_master)}")
print(f"Índice único: {df_master.index.is_unique}")
df_master.head()

### 6.6. Diagnóstico de la integración

Cobertura del matching por nivel y visitas con dirección que quedaron sin emparejar (candidatas a revisión manual o a ajustar los umbrales de `build_match_table_v2`).

In [ ]:
matched = match_table["sitio_id"].notna()

print(f"Visitas emparejadas:    {matched.sum()} / {len(match_table)}")
print(f"Sitios EDAN con visita: {match_table.loc[matched, 'sitio_id'].nunique()} / {df_edan['sitio_id'].nunique()}")
print()
print(match_table.loc[matched, "match_method"].value_counts().to_string())

# Unmatched visitas that DO have an address — candidates for manual review
# or for loosening vector_tol / fuzzy_threshold
sin_match = df_visitas.merge(match_table.loc[~matched, ["visita_id"]], on="visita_id")
sin_match = sin_match[sin_match["direccion_norm"].astype(str).str.strip().ne("")]
print(f"\nVisitas con dirección pero sin match: {len(sin_match)}")
sin_match[["visita_id", "direccion_norm"]].head(20)

### 6.7. Resultado final: `df_integrado`

Dataset integrado definitivo. Parte de `df_master` (merge externo, nada se pierde) y agrega:

- **`fuente`** — origen de cada registro: `edan+visita` (emparejado con trust ≥ 0.70), `solo_edan`, `solo_visita`.
- **`trust_score`** — confiabilidad del registro unificado, en [0, 1]:
  - Registros **emparejados** heredan el trust del match (base por método + corroboración semántica/barrio/geo de 6.4). Por el corte de 6.4, siempre ≥ 0.70.
  - Registros de **una sola fuente** no pueden verificarse cruzadamente → puntaje de completitud con techo 0.45: base 0.20, +0.10 dirección parseable, +0.10 coordenadas utilizables, +0.05 barrio presente.
- **Variables numéricas unificadas con `max()`** — `n_fallecidos_total`, `n_atrapamientos_total`, `n_rescatados_total`. Se toma el **máximo** entre EDAN y visita, no la suma: ambas fuentes describen el mismo evento en el mismo predio, por lo que sumar duplicaría personas contadas dos veces; el máximo es la estimación conservadora de "al menos N". Queda `NaN` solo si ninguna fuente reporta (ausencia ≠ cero).
- **Coordenadas estandarizadas semánticamente** — las coordenadas NO se normalizan con nomenclatura IGAC: IGAC codifica nomenclatura VIAL (tipos de vía, numeración catastral) y no aplica a posiciones geográficas; forzarlo sería contraproducente. El estándar correcto para coordenadas es **WGS84 decimal**: `parse_latlon` (sección 3.3) interpreta los formatos heterogéneos de origen (grados decimales, pares invertidos, texto con ruido), valida contra el bounding box de Cali y aquí se materializa en `lat` / `lon` numéricas + `coords_unificadas` reescrita al formato uniforme `"lat, lon"` con 6 decimales (~0.1 m de precisión).
- **Columnas de texto unificadas** (`direccion_unificada`, `barrio_unificado`, `comuna_unificada`) — coalescencia con prioridad EDAN y fallback a la visita; los sufijos `_edan` / `_visita` se conservan para auditoría.


In [ ]:
# df_integrado: final integrated dataset — one row per unique record
# (EDAN site + matched visita | EDAN-only | visita-only).
# Adds source flag, trust_score, max-consolidated shared numeric metrics,
# WGS84-standardized coordinates and coalesced text columns while keeping
# every original column intact.
df_integrado = df_master.copy()

fuente = np.select(
    [df_integrado["sitio_id"].notna() & df_integrado["visita_id"].notna(),
     df_integrado["sitio_id"].notna()],
    ["edan+visita", "solo_edan"],
    default="solo_visita",
)
df_integrado.insert(0, "fuente", fuente)

def _coalesce(col_edan, col_visita):
    a = df_integrado[col_edan].astype("object")
    b = df_integrado[col_visita].astype("object")
    a_empty = a.isna() | a.astype(str).str.strip().isin(["", "-", "nan", "None"])
    return a.mask(a_empty, b)

# Unified text columns (suffixed originals stay available for auditing)
for pos, (name, col_edan, col_visita) in enumerate([
    ("direccion_unificada", "direccion_norm_edan", "direccion_norm_visita"),
    ("barrio_unificado", "barrio_vereda_edan", "barrio_vereda_visita"),
    ("comuna_unificada", "comuna_corregimiento_edan", "comuna_corregimiento_visita"),
], start=1):
    df_integrado.insert(pos, name, _coalesce(col_edan, col_visita))

# Coordinates: semantic standardization to WGS84 decimal degrees.
# IGAC normalization applies to road nomenclature, not to geographic
# positions — the correct standard here is parsing every source format
# into validated (lat, lon) within the Cali bounding box.
_pts_e = df_integrado["coords_edan"].map(parse_latlon)
_pts_v = df_integrado["coords_visita"].map(parse_latlon)
_pts = _pts_e.where(_pts_e.notna(), _pts_v)
df_integrado.insert(4, "lat", _pts.map(lambda p: round(p[0], 6) if p else np.nan))
df_integrado.insert(5, "lon", _pts.map(lambda p: round(p[1], 6) if p else np.nan))
df_integrado.insert(6, "coords_unificadas",
                    _pts.map(lambda p: f"{p[0]:.6f}, {p[1]:.6f}" if p else None))

# Shared numeric metrics: both sources describe the SAME event on the same
# site, so the unified value is the MAX of both sides (summing would double
# count people reported by both). NaN only when neither source reports.
def _as_num(col):
    s = df_integrado[col].astype(str).str.strip().str.replace(",", "", regex=False)
    return pd.to_numeric(s.str.extract(r"(-?\d+\.?\d*)", expand=False), errors="coerce")

SHARED_NUMERIC = ["n_fallecidos", "n_atrapamientos", "n_rescatados"]
for name in SHARED_NUMERIC:
    df_integrado[f"{name}_total"] = np.fmax(_as_num(f"{name}_edan"),
                                            _as_num(f"{name}_visita"))

# trust_score per record in [0, 1]:
#  - matched records inherit the match trust (>= 0.70 by the 6.4 cutoff)
#  - single-source records get a completeness score capped at 0.45: they
#    exist in only one source, so they cannot be cross-verified
def _single_source_trust(row) -> float:
    t = 0.20
    addr = canonicalize_for_match(row["direccion_unificada"])
    if addr and address_to_vector(addr)[0] != 0:
        t += 0.10   # parseable address
    if row["coords_unificadas"]:
        t += 0.10   # usable coordinates
    if str(row["barrio_unificado"]).strip() not in {"", "-", "nan", "None"}:
        t += 0.05   # barrio present
    return round(t, 2)

trust_score = df_integrado["trust"].where(
    df_integrado["trust"].notna(),
    df_integrado.apply(_single_source_trust, axis=1),
)
df_integrado.insert(1, "trust_score", trust_score.astype(float))

print(f"df_integrado: {df_integrado.shape[0]} registros x {df_integrado.shape[1]} columnas")
print(df_integrado["fuente"].value_counts().to_string())
print(f"\ncoordenadas estandarizadas (WGS84 decimal): {int(df_integrado['lat'].notna().sum())} registros")
print("\ntrust_score por fuente:")
print(df_integrado.groupby("fuente")["trust_score"]
      .agg(["count", "mean", "min", "median", "max"]).round(2).to_string())
print("\ntotales unificados (max entre EDAN y visita):")
for name in SHARED_NUMERIC:
    a, b = _as_num(f"{name}_edan"), _as_num(f"{name}_visita")
    t = df_integrado[f"{name}_total"]
    print(f"  {name}_total = {t.sum():.0f}  (edan {a.sum():.0f}, visita {b.sum():.0f}, "
          f"no-nulos {int(t.notna().sum())})")
df_integrado.head()


### 6.8. Subconjuntos de entrega: `df_match_lm` y `df_integrado_confiable`

Dos vistas finales sobre `df_integrado`:

- **`df_match_lm`** — solo los registros cuyo emparejamiento lo produjo el **modelo de lenguaje** (`match_method == "embedding"`): pares que ningún método estructural (huella, vector, esquina, geo, TF-IDF, fuzzy) pudo resolver y que el LM rescató vía semántica + guard numérico. Es la contribución neta del LM al resultado.
- **`df_integrado_confiable`** — todos los registros con `trust_score > 0.7` (en la práctica, los emparejados que sobrevivieron el corte de 6.4: los de una sola fuente tienen techo 0.45). Versión **limpia para consumo**: se eliminan las columnas redundantes y las variables sintéticas intermedias que ya cumplieron su función:
  - Redundantes con las unificadas: `direccion_norm_*`, `barrio_vereda_*`, `comuna_corregimiento_*`, `coords_*`, `n_fallecidos_*`, `n_atrapamientos_*`, `n_rescatados_*` (sus valores viven en `*_unificada` / `*_total`).
  - Sintéticas de trabajo del pipeline: `integration_handshake_*` (huella de matching), `match_score` (superada por `trust_score`), `trust` (duplicada como `trust_score`).
  - Las columnas **reales** de origen (`direccion_edan`, `direccion_visita`, descripciones, observaciones, etc.) se conservan — solo renombradas por el merge con sufijos, nunca destruidas. `df_integrado` completo sigue disponible para auditoría.


In [ ]:
# df_match_lm: records whose pairing was produced solely by the language
# model tier — the LM's net contribution to the integration.
df_match_lm = df_integrado[df_integrado["match_method"] == "embedding"].copy()
print(f"df_match_lm (matches aportados por el LM): {len(df_match_lm)} registros")
print(df_match_lm[["trust_score", "direccion_norm_edan", "direccion_norm_visita",
                   "barrio_unificado"]].to_string(max_colwidth=40))

# df_integrado_confiable: high-trust records only (trust_score > 0.7),
# with redundant and spent synthetic columns removed. Real source columns
# (raw addresses, descriptions, observations) are kept; the full
# df_integrado remains available for auditing.
_REDUNDANT = [
    # superseded by the unified/consolidated columns
    "direccion_norm_edan", "direccion_norm_visita",
    "barrio_vereda_edan", "barrio_vereda_visita",
    "comuna_corregimiento_edan", "comuna_corregimiento_visita",
    "coords_edan", "coords_visita",
    "n_fallecidos_edan", "n_fallecidos_visita",
    "n_atrapamientos_edan", "n_atrapamientos_visita",
    "n_rescatados_edan", "n_rescatados_visita",
    # spent pipeline internals
    "integration_handshake_edan", "integration_handshake_visita",
    "match_score", "trust",
]
df_integrado_confiable = (
    df_integrado[df_integrado["trust_score"] > 0.7]
    .drop(columns=_REDUNDANT, errors="ignore")
    .copy()
)

print(f"\ndf_integrado_confiable: {df_integrado_confiable.shape[0]} registros x "
      f"{df_integrado_confiable.shape[1]} columnas "
      f"({df_integrado.shape[1] - df_integrado_confiable.shape[1]} columnas eliminadas)")
print(df_integrado_confiable["fuente"].value_counts().to_string())
print("\ntrust_score:")
print(df_integrado_confiable["trust_score"].describe().round(2).to_string())
print("\nmatch_method:")
print(df_integrado_confiable["match_method"].value_counts().to_string())
df_integrado_confiable.head()


### 6.9. Función de exportación para revisión manual

`exportar_match_lm()` genera **`revision_match_lm.xlsx`** con los matches producidos por el modelo de lenguaje: direcciones de ambas fuentes lado a lado, señales de corroboración y dos columnas vacías (`verificado`, `observaciones_revision`) para registrar el veredicto. Acepta cualquier subconjunto de `df_integrado` vía el parámetro `df` (p. ej. los matches de menor trust) y una ruta de salida con `path`.

In [ ]:
# Excel export of the LM-produced matches for manual review.
import os

REVIEW_COLS = [
    "trust_score", "match_score",
    "direccion_edan", "direccion_norm_edan",
    "direccion_visita", "direccion_norm_visita",
    "barrio_vereda_edan", "barrio_vereda_visita",
    "comuna_unificada", "coords_unificadas",
    "nombre_estructura", "tipo_estructura_edan", "tipo_estructura_visita",
]

def exportar_match_lm(df=None, path="revision_match_lm.xlsx"):
    """
    Export the LM (embedding) matches to an Excel file for manual review.

    Layout is reviewer-oriented: both source addresses side by side,
    corroborating signals, and two empty columns (verificado /
    observaciones_revision) to record the verdict per match.

    Parameters
    ----------
    df : DataFrame, optional. Defaults to df_match_lm; pass any subset of
         df_integrado (e.g. low-trust matches) to review other slices.
    path : str, output .xlsx path.
    """
    if df is None:
        df = df_match_lm
    export = df[[c for c in REVIEW_COLS if c in df.columns]].copy()
    export.insert(0, "verificado", "")            # reviewer fills: SI / NO
    export.insert(1, "observaciones_revision", "")

    # If the target file is open in Excel, Windows locks it — fall back to a
    # numbered name instead of aborting the whole notebook run.
    base, ext = os.path.splitext(path)
    for attempt in range(10):
        candidate = path if attempt == 0 else f"{base}_{attempt}{ext}"
        try:
            with open(candidate, "ab"):
                pass
            path = candidate
            break
        except PermissionError:
            continue

    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        export.to_excel(writer, sheet_name="matches_lm", index=True,
                        index_label="registro_id")
        ws = writer.sheets["matches_lm"]
        ws.freeze_panes = "B2"                     # keep header + id visible
        for col_cells in ws.columns:
            width = max(len(str(c.value)) if c.value is not None else 0
                        for c in col_cells)
            ws.column_dimensions[col_cells[0].column_letter].width = \
                min(max(width + 2, 12), 55)

    print(f"Exportado: {path}  ({len(export)} matches para revisión)")
    return path

exportar_match_lm()


### 6.10. Persistencia de IDs en las hojas de origen

Escribe `sitio_id` (hoja EDAN) y `visita_id` (hoja de visitas) como **última columna** de cada Google Sheet de origen, para poder ubicar cada registro del análisis en su fila original. Contrato:

- **La columna persistida es la fuente de verdad.** Las secciones 4/5 la reutilizan al leer: un registro conserva su ID aunque la hoja se reordene o se inserten filas. Solo las filas sin ID (respuestas nuevas del formulario) reciben uno, generado con la misma secuencia determinista (semillas 42/88) evitando colisiones con los existentes.
- La columna se agrega al final del header, o se **reutiliza** si ya existe — la celda es repetible corrida tras corrida.
- Una celda no vacía **jamás se sobreescribe** y la escritura es un único update de rango de una sola columna: el resto de la hoja queda intacto por construcción.
- **Compuerta de consistencia**: si algún ID usado por esta corrida no queda representado en la columna final (hoja modificada a mitad de corrida), la función aborta sin escribir.

In [ ]:
# Persist the notebook IDs back to the source Google Sheets as a LAST
# column, so every record can be located later. Safety by construction:
#  - the ID column is appended at the END of the header, or reused if it
#    already exists (repeatable run after run)
#  - a non-empty cell is NEVER overwritten: existing IDs are the source of
#    truth (sections 4/5 read them back), only empty cells receive a new ID
#  - the write is a single one-column range update — no other cell can change
#  - consistency gate: every ID used by this notebook run must exist in the
#    final column, otherwise the function aborts without writing
from gspread.utils import rowcol_to_a1

def escribir_ids_en_sheet(spreadsheet_id, sheet_name, col_name, seed,
                          ids_notebook, dry_run=False):
    """Append/refresh the persisted ID column on a source sheet (cell notes)."""
    creds_rw = Credentials.from_service_account_file(
        SERVICE_ACCOUNT_FILE,
        scopes=["https://www.googleapis.com/auth/spreadsheets"],
    )
    ws = gspread.authorize(creds_rw).open_by_key(spreadsheet_id).worksheet(sheet_name)
    vals = ws.get_all_values()
    header, data = vals[0], vals[1:]
    n_rows = len(data)

    if col_name in header:
        col = header.index(col_name) + 1
    else:
        col = len(header) + 1
        if not dry_run and ws.col_count < col:
            ws.add_cols(col - ws.col_count)

    current = [(r[col - 1].strip() if len(r) >= col else "") for r in data]
    taken = {c for c in current if c}
    fill = iter(generar_ids(seed, sum(1 for c in current if not c), taken))
    new_col = [col_name] + [c if c else next(fill) for c in current]

    faltantes = set(ids_notebook) - set(new_col[1:])
    if faltantes:
        raise RuntimeError(
            f"[{sheet_name}] {len(faltantes)} IDs del notebook no estan en la "
            f"columna final (hoja modificada durante la corrida) — NO se escribe.")

    escritos = sum(1 for c in current if not c)
    rango = f"{rowcol_to_a1(1, col)}:{rowcol_to_a1(n_rows + 1, col)}"
    print(f"[{sheet_name}] columna '{col_name}' -> {rango.split(':')[0][:-1]} | "
          f"{n_rows} filas | nuevos: {escritos} | conservados: {n_rows - escritos}")
    if dry_run:
        print("  dry_run=True — no se escribio nada.")
        return
    ws.update(values=[[v] for v in new_col], range_name=rango,
              value_input_option="RAW")
    print(f"  Escrito OK ({len(new_col)} celdas de la columna, nada mas).")

escribir_ids_en_sheet(
    "1QRLezOtMTZpePluDl7VVzxINYLgvACVB76z54vJNpfE",
    "EDAN 100826 - Datos Madre", "sitio_id", 42,
    df_edan["sitio_id"].tolist(),
)
escribir_ids_en_sheet(
    "1SIzarDbjtaD6JVM7cUHWqrcLBJNgYj6ZooHVU9tRKN4",
    "Respuestas de formulario 1", "visita_id", 88,
    df_visitas["visita_id"].tolist(),
)